# Como-Article Ground/Vehicle Full Corpus

Spec: `docs/specs/0006-como-article-ground-vehicle-mvp.md`

This notebook runs the same optimized extraction workflow as the single-shard MVP, but defaults to all local brWaC clean shards and a separate full-corpus bronze path so results can be compared without overwriting the baseline or half-corpus bronze datasets.

```text
Full-corpus bronze segments
  -> native Spark rlike prefilter for curated GROUND como article
  -> Python UDF extraction only on plausible rows
  -> gold Parquet and WebApp-ready CSV outputs
```


In [1]:
import os
from pathlib import Path
import sys

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src/tal_qual").exists():
            return candidate
        if (candidate / "work/src/tal_qual").exists():
            return candidate / "work"
    raise RuntimeError(f"Could not find repository root from {start}")


REPO_ROOT = find_repo_root(Path.cwd())
os.chdir(REPO_ROOT)
SRC_DIR = REPO_ROOT / "src"
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

REPO_ROOT


PosixPath('/home/jovyan/work')

In [2]:
import os
import tempfile
import zipfile

from pyspark.sql import SparkSession

SPARK_MASTER = os.environ.get("TAL_QUAL_SPARK_MASTER", "local[4]")
SPARK_PARALLELISM = os.environ.get("TAL_QUAL_SPARK_PARALLELISM", "4")
SPARK_SHUFFLE_PARTITIONS = os.environ.get("TAL_QUAL_SPARK_SHUFFLE_PARTITIONS", "4")

spark = (
    SparkSession.builder
    .master(SPARK_MASTER)
    .appName("tal-qual-como-article-ground-vehicle-full-corpus")
    .config("spark.default.parallelism", SPARK_PARALLELISM)
    .config("spark.sql.shuffle.partitions", SPARK_SHUFFLE_PARTITIONS)
    .config("spark.sql.files.maxPartitionBytes", str(256 * 1024 * 1024))
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

package_zip = Path(tempfile.gettempdir()) / "tal_qual_src.zip"
with zipfile.ZipFile(package_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for file_path in sorted((SRC_DIR / "tal_qual").rglob("*.py")):
        archive.write(file_path, file_path.relative_to(SRC_DIR))

spark.sparkContext.addPyFile(str(package_zip))
SPARK_MASTER, spark.version


('local[4]', '4.1.1')

In [3]:
from tal_qual.bronze import BRONZE_OUTPUT_PATH, RAW_CORPUS_INPUT, load_or_build_bronze_dataframe
from tal_qual.como_article_ground_vehicle import (
    COMO_ARTICLE_EXAMPLES_OUTPUT_PATH,
    COMO_ARTICLE_GROUND_COUNTS_OUTPUT_PATH,
    COMO_ARTICLE_GROUND_VEHICLE_COUNTS_OUTPUT_PATH,
    COMO_ARTICLE_GROUND_VEHICLE_OUTPUT_PATH,
    COMO_ARTICLE_REVIEW_SAMPLE_OUTPUT_PATH,
    COMO_ARTICLE_VEHICLE_COUNTS_OUTPUT_PATH,
    COMO_ARTICLE_VEHICLE_GROUND_COUNTS_OUTPUT_PATH,
    como_article_examples_dataframe,
    como_article_ground_counts_dataframe,
    como_article_ground_vehicle_counts_dataframe,
    como_article_review_sample_dataframe,
    como_article_vehicle_counts_dataframe,
    como_article_vehicle_ground_counts_dataframe,
    prefilter_como_article_ground_vehicle_bronze_dataframe,
    prepare_como_article_ground_vehicle_dataframe,
    write_como_article_csv_outputs,
    write_como_article_ground_vehicle_parquet,
)


## Load Or Build Bronze

The extractor expects the bronze schema from notebook 02. If bronze output is absent, this notebook builds it from the configured raw shard.


In [4]:
FULL_CORPUS_RAW_INPUT = "data/raw/brwac-clean-*.txt.gz"
FULL_CORPUS_BRONZE_PATH = "data/bronze/brwac_segments_full"

bronze_path = REPO_ROOT / os.environ.get("TAL_QUAL_BRONZE_PATH", FULL_CORPUS_BRONZE_PATH)
raw_path = REPO_ROOT / os.environ.get("TAL_QUAL_RAW_CORPUS_INPUT", FULL_CORPUS_RAW_INPUT)

bronze_df = load_or_build_bronze_dataframe(spark, raw_path, bronze_path)

print(f"Raw input: {raw_path}")
print(f"Bronze path: {bronze_path}")
bronze_df.select("source_file").distinct().orderBy("source_file").show(20, truncate=False)

bronze_df.printSchema()


Raw input: /home/jovyan/work/data/raw/brwac-clean-*.txt.gz
Bronze path: /home/jovyan/work/data/bronze/brwac_segments_full
+-------------------------------------------------------+
|source_file                                            |
+-------------------------------------------------------+
|file:///home/jovyan/work/data/raw/brwac-clean-1.txt.gz |
|file:///home/jovyan/work/data/raw/brwac-clean-10.txt.gz|
|file:///home/jovyan/work/data/raw/brwac-clean-11.txt.gz|
|file:///home/jovyan/work/data/raw/brwac-clean-12.txt.gz|
|file:///home/jovyan/work/data/raw/brwac-clean-2.txt.gz |
|file:///home/jovyan/work/data/raw/brwac-clean-3.txt.gz |
|file:///home/jovyan/work/data/raw/brwac-clean-4.txt.gz |
|file:///home/jovyan/work/data/raw/brwac-clean-5.txt.gz |
|file:///home/jovyan/work/data/raw/brwac-clean-6.txt.gz |
|file:///home/jovyan/work/data/raw/brwac-clean-7.txt.gz |
|file:///home/jovyan/work/data/raw/brwac-clean-8.txt.gz |
|file:///home/jovyan/work/data/raw/brwac-clean-9.txt.gz |
+-------

## Prefilter And Extract Spec-0006 Candidates

The native prefilter is intentionally broad but cheap. It reduces the number of rows sent through the Python UDF.


In [5]:
prefiltered_bronze_df = prefilter_como_article_ground_vehicle_bronze_dataframe(bronze_df).persist()
prefiltered_count = prefiltered_bronze_df.count()
print(f"Prefiltered bronze rows: {prefiltered_count:,}")

candidates_df = prepare_como_article_ground_vehicle_dataframe(prefiltered_bronze_df).persist()
candidate_count = candidates_df.count()
print(f"Spec-0006 candidates: {candidate_count:,}")
candidates_df


Prefiltered bronze rows: 3,524


/usr/local/spark/python/pyspark/sql/udf.py:134: UserWarning: Cannot infer the eval type from type hints. 
  warnings.warn("Cannot infer the eval type from type hints. ", UserWarning)


Spec-0006 candidates: 2,136


DataFrame[candidate_id: string, source_file: string, original_line_id: bigint, segment_id: int, pattern_type: string, connector: string, connector_text: string, matched_text: string, candidate_full_text: string, text_before: string, tenor_text: string, tenor_lemma: string, tenor_confidence: string, ground_text: string, ground_lemma: string, ground_type: string, ground_source: string, vehicle_text: string, vehicle_lemma: string, vehicle_head: string, vehicle_head_lemma: string, vehicle_phrase_length_tokens: int, filter_label: string, reject_reason: string, confidence: double, needs_review: boolean, char_start: int, char_end: int, connector_start: int, connector_end: int, vehicle_start: int, vehicle_end: int]

In [6]:
candidates_df.printSchema()


root
 |-- candidate_id: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- original_line_id: long (nullable = true)
 |-- segment_id: integer (nullable = true)
 |-- pattern_type: string (nullable = true)
 |-- connector: string (nullable = true)
 |-- connector_text: string (nullable = true)
 |-- matched_text: string (nullable = true)
 |-- candidate_full_text: string (nullable = true)
 |-- text_before: string (nullable = true)
 |-- tenor_text: string (nullable = true)
 |-- tenor_lemma: string (nullable = true)
 |-- tenor_confidence: string (nullable = true)
 |-- ground_text: string (nullable = true)
 |-- ground_lemma: string (nullable = true)
 |-- ground_type: string (nullable = true)
 |-- ground_source: string (nullable = true)
 |-- vehicle_text: string (nullable = true)
 |-- vehicle_lemma: string (nullable = true)
 |-- vehicle_head: string (nullable = true)
 |-- vehicle_head_lemma: string (nullable = true)
 |-- vehicle_phrase_length_tokens: integer (nullable = true

## Write Gold Dataset And Visualization CSVs


In [7]:
write_como_article_ground_vehicle_parquet(candidates_df, REPO_ROOT / COMO_ARTICLE_GROUND_VEHICLE_OUTPUT_PATH)
write_como_article_csv_outputs(
    candidates_df,
    ground_vehicle_counts_path=REPO_ROOT / COMO_ARTICLE_GROUND_VEHICLE_COUNTS_OUTPUT_PATH,
    vehicle_ground_counts_path=REPO_ROOT / COMO_ARTICLE_VEHICLE_GROUND_COUNTS_OUTPUT_PATH,
    ground_counts_path=REPO_ROOT / COMO_ARTICLE_GROUND_COUNTS_OUTPUT_PATH,
    vehicle_counts_path=REPO_ROOT / COMO_ARTICLE_VEHICLE_COUNTS_OUTPUT_PATH,
    examples_path=REPO_ROOT / COMO_ARTICLE_EXAMPLES_OUTPUT_PATH,
    review_sample_path=REPO_ROOT / COMO_ARTICLE_REVIEW_SAMPLE_OUTPUT_PATH,
)


## Inspect Visualization Tables

These displays use small materialized pandas tables for readability. The durable CSV outputs above remain the source of truth for downstream visualization work.


In [9]:
from pyspark.sql.functions import col

def display_table(spark_df, limit=30):
    display(spark_df.limit(limit).toPandas())



### Evaluation Strategy Outputs


In [10]:

print(f"Evaluation dataset: {EVALUATION_DATASET_PATH}")
print(f"Evaluation outputs: {EVALUATION_OUTPUT_ROOT}")
display_table(
    evaluation_df.groupBy("pattern_type").count().orderBy("pattern_type"),
    limit=20,
)


Evaluation dataset: data/gold/evaluation/como_article_strategy_candidates
Evaluation outputs: outputs/evaluation


,pattern_type,count
0,como_article_ground_vehicle,2136


### Ground -> Vehicle Pairs


In [11]:
display_table(
    como_article_ground_vehicle_counts_dataframe(candidates_df).select(
        col("ground_lemma").alias("ground"),
        col("vehicle_head_lemma").alias("vehicle_head"),
        "count",
    ),
    limit=40,
)


,ground,vehicle_head,count
0,cair,luva,452
1,cair,bomba,208
2,encaixar,luva,37
3,caber,luva,25
4,duro,pedra,25
5,leve,pluma,21
6,rápido,raio,20
7,assentar,luva,19
8,duro,rocha,17
9,forte,touro,17


### Vehicle -> Ground Pairs


In [12]:
display_table(
    como_article_vehicle_ground_counts_dataframe(candidates_df).select(
        col("vehicle_head_lemma").alias("vehicle_head"),
        col("ground_lemma").alias("ground"),
        "count",
    ),
    limit=40,
)


,vehicle_head,ground,count
0,luva,cair,452
1,bomba,cair,208
2,luva,encaixar,37
3,luva,caber,25
4,pedra,duro,25
5,pluma,leve,21
6,raio,rápido,20
7,luva,assentar,19
8,rocha,duro,17
9,touro,forte,17


### Ground Counts


In [13]:
display_table(como_article_ground_counts_dataframe(candidates_df), limit=40)


,ground_lemma,count
0,cair,783
1,trabalhar,167
2,forte,97
3,duro,83
4,rápido,82
5,correr,70
6,leve,70
7,grande,64
8,brilhar,61
9,encaixar,52


### Vehicle Counts


In [14]:
display_table(como_article_vehicle_counts_dataframe(candidates_df), limit=40)


,pattern_type,vehicle_head_clean,count
0,como_article_ground_vehicle,luva,533
1,como_article_ground_vehicle,bomba,210
2,como_article_ground_vehicle,raio,38
3,como_article_ground_vehicle,pedra,35
4,como_article_ground_vehicle,pássaro,34
5,como_article_ground_vehicle,rocha,29
6,como_article_ground_vehicle,pluma,22
7,como_article_ground_vehicle,touro,18
8,como_article_ground_vehicle,flecha,14
9,como_article_ground_vehicle,borboleta,13


### Examples


In [15]:
display_table(
    como_article_examples_dataframe(candidates_df, limit=100).select(
        "ground_text",
        "connector_text",
        "vehicle_text",
        "tenor_text",
        "candidate_full_text",
    ),
    limit=50,
)


,pattern_type,ground_text,connector_text,vehicle_text_clean,candidate_full_text
0,como_article_ground_vehicle,Alegre,como uma,alternativa ao pós-Socrates,Eu não olho para Manuel Alegre como uma altern...
1,como_article_ground_vehicle,alegre,como uma,amiga minha,"u quero ir a Paris, tomar uns ""banhos"" e volta..."
2,como_article_ground_vehicle,Alegre,como um,caçador ou arranjador de votos,"a, criativa, produtora de nova realidade, que ..."
3,como_article_ground_vehicle,Alegre,como uma,cidade conectada,As imagens apresentam o cotidiano de Porto Ale...
4,como_article_ground_vehicle,alegre,como uma,criança,Saí da jaula dos filhotes de leões alegre como...
5,como_article_ground_vehicle,alegre,como uma,criança ao ter comprado,"epois passa-me a nostalgia, pois só o discerni..."
6,como_article_ground_vehicle,alegre,como um,galo,", e, por ter ela crescido, em tão estranhos «v..."
7,como_article_ground_vehicle,alegre,como um,horto de flores,- Vila Viçosa é alegre como um horto de flores
8,como_article_ground_vehicle,alegre,como uma,manhã de abril,"Malvina, que vinha do interior da casa, risonh..."
9,como_article_ground_vehicle,Alegre,como um,modelo internacional,ferência internacional em políticas sociais e ...


### Review Sample


In [16]:
display_table(
    como_article_review_sample_dataframe(candidates_df, limit=100).select(
        "ground_type",
        "ground_text",
        "connector_text",
        "vehicle_text",
        "tenor_text",
        "confidence",
        "needs_review",
        "candidate_full_text",
    ),
    limit=50,
)


,pattern_type,ground_text,connector_text,vehicle_text_clean,quality_reason,needs_review,candidate_full_text
0,como_article_ground_vehicle,Mole,como uma,carne de sol,spec_0006_strict_filter,True,Mole como uma carne de sol
1,como_article_ground_vehicle,Pesado,como um,elefante luz,spec_0006_strict_filter,True,Pesado como um elefante luz
2,como_article_ground_vehicle,Caiu,como uma,luva,spec_0006_strict_filter,True,Caiu como uma luva
3,como_article_ground_vehicle,Correu,como um,rastilho de pólvora,spec_0006_strict_filter,True,Correu como um rastilho de pólvora
4,como_article_ground_vehicle,Caiu,como uma,luva,spec_0006_strict_filter,True,Caiu como uma luva
5,como_article_ground_vehicle,Grande,como um,continente,spec_0006_strict_filter,True,Grande como um continente
6,como_article_ground_vehicle,Trabalhar,como uma,aranha,spec_0006_strict_filter,True,Trabalhar como uma aranha
7,como_article_ground_vehicle,Pequeno,como um,tostão,spec_0006_strict_filter,True,Pequeno como um tostão
8,como_article_ground_vehicle,Rápido,como uma,raposa,spec_0006_strict_filter,True,Rápido como uma raposa
9,como_article_ground_vehicle,Correu,como uma,bala para a pensão,spec_0006_strict_filter,True,Correu como uma bala para a pensão
